In [1]:
import torch
import torch.nn.functional as F
from diffusers import StableDiffusionPipeline
from transformers import CLIPTokenizer, CLIPTextModel

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(device)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


In [2]:
model_id = "openai/clip-vit-large-patch14"
tokenizer = CLIPTokenizer.from_pretrained(model_id)
text_encoder = CLIPTextModel.from_pretrained(model_id).to(device)

In [3]:
# Function to get text embedding
def get_text_embedding(prompt: str):
    inputs = tokenizer(
        prompt,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        outputs = text_encoder(**inputs)
        # You might get different outputs: last_hidden_state, or pooled output
        # For SD v1.5, the documentation says “non-pooled output of the text encoder is fed into the UNet” :contentReference[oaicite:3]{index=3}
        embedding = outputs.last_hidden_state  # shape: (batch_size, seq_len, hidden_dim)
    return embedding

In [17]:
# Example prompts
emb1 = get_text_embedding("Herd of cows on alpine pasture among mountains in Alps, northern Italy. Stock Photo")
# emb2 = get_text_embedding("a photograph of an astronaut riding a horse")

# Simplify to one vector each (e.g., average over tokens)
# Note: Depending on how you define similarity, you might use e.g. the embedding[:,0,:] token or mean over seq_len.
vec1 = emb1.mean(dim=1)  # shape: (batch_size, hidden_dim)
# vec2 = emb2.mean(dim=1)
vec2 = mapped.mean(dim=1)  # shape: (batch_size, hidden_dim)

In [18]:
print(emb1.shape, mapped.shape)
print(vec1.shape, vec2.shape)

torch.Size([1, 77, 768]) torch.Size([1, 77, 768])
torch.Size([1, 768]) torch.Size([1, 768])


In [19]:
vec1 = F.normalize(vec1, p=2, dim=1)
vec2 = F.normalize(vec2, p=2, dim=1)
similarity = torch.matmul(vec1, vec2.T)  # shape: (batch_size, batch_size)
print(f"Cosine similarity: {similarity.item():.4f}")

Cosine similarity: 0.3563


In [13]:
from pathlib import Path
from mapper_model import ImageToTextMapper

def _load_mapper(mapper_path: str, device: torch.device):
    data = torch.load(mapper_path, map_location="cpu")
    cfg = data["config"]
    mapper = ImageToTextMapper(in_dim=cfg["in_dim"], out_seq_len=cfg["out_seq_len"], out_dim=cfg["out_dim"], num_layers=4)
    state = data["state_dict"]
    # strip DataParallel prefix if present
    if any(k.startswith("module.") for k in list(state.keys())):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}
    mapper.load_state_dict(state)
    mapper.to(device).eval()
    return mapper, cfg

mapper_path = Path("/home1/koustav/Image_to_Image_Diffusion/mapper_ckpt_new/mapper_epoch24.pth")

mapper, cfg = _load_mapper(mapper_path, device=torch.device("cpu"))  # load to cpu then move below
mapper.to(device).eval()

ImageToTextMapper(
  (net): Sequential(
    (0): Linear(in_features=768, out_features=4096, bias=True)
    (1): GELU(approximate='none')
    (2): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=4096, out_features=4096, bias=True)
    (5): GELU(approximate='none')
    (6): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=4096, out_features=4096, bias=True)
    (9): GELU(approximate='none')
    (10): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
    (11): Dropout(p=0.1, inplace=False)
    (12): Linear(in_features=4096, out_features=59136, bias=True)
  )
)

In [14]:
from torchvision import transforms
from PIL import Image
def _prepare_image(img_path: str):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((512, 512))
    preprocess = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    t = preprocess(img).unsqueeze(0)  # [1,C,H,W]
    return img, t

input_image_path = "/home1/koustav/Image_to_Image_Diffusion/output.png"
pil_img, img_tensor = _prepare_image(input_image_path)
img_tensor = img_tensor.to(device=device, dtype=torch.float32)

In [15]:
from transformers import CLIPProcessor, CLIPModel, CLIPTokenizer, CLIPTextModel

clip_model_name = "openai/clip-vit-large-patch14"

processor = CLIPProcessor.from_pretrained(clip_model_name, use_fast=True)
clip = CLIPModel.from_pretrained(clip_model_name).to(device).eval()


with torch.no_grad():
    # CLIP image features -> mapper conditioning
    clip_inputs = processor(images=pil_img, return_tensors="pt")
    clip_inputs = {k: v.to(device) for k, v in clip_inputs.items()}
    img_feats = clip.get_image_features(**clip_inputs)  # float32 on device
    mapped = mapper(img_feats)  # [1, L, H] float32

In [16]:
mapped.shape, img_feats.shape

(torch.Size([1, 77, 768]), torch.Size([1, 768]))

In [39]:

tvec1 = F.normalize(mapped.mean(dim=1), p=2, dim=1)
tvec2 = F.normalize(img_feats, p=2, dim=1)
similarity = torch.matmul(tvec1, tvec2.T)  # shape: (batch_size, batch_size)

print(similarity)


tensor([[0.1066]], device='cuda:0')
